# NO SE PUEDE HACER PORQUE NO DISPONEMOS DEL CONJUNTO DE DATOS ORIGINAL SIN RUIDO PARA PODER ENTRENAR EL AUTOENCODER

In [11]:
# # Description
# 
# Prove that the Noise Reduction model works on a 2D polinomial.

# Add libs folder path to system variables to make the importation of local libraries easier.
# __________________________________________________________________________________________________
import os
import sys


CURRENT_DIR = os.getcwd()
FOLDERS = CURRENT_DIR.split(os.sep)
TESIS_FOLDER_INDEX = FOLDERS.index('S-noise-gradient')
CURRENT_DIR = os.sep.join(FOLDERS[:TESIS_FOLDER_INDEX+1])
LIBS_PATH = os.path.join(CURRENT_DIR, 'src', 'libs')
assert os.path.exists(LIBS_PATH)
sys.path.append(LIBS_PATH)


# Import external and local libraries
# __________________________________________________________________________________________________
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import json
from typing import List

# Locals
from utils import *
from dataset import TensorDataset
from models import *
import config as cfg
from dlnr import DLNoiseReduction


# Global / Path variables
# __________________________________________________________________________________________________
DATA_PATH = os.path.join(CURRENT_DIR, 'data')
CHECKPOINT_PATH = os.path.join(CURRENT_DIR, 'checkpoints')
CONFIG_PATH = os.path.join(CURRENT_DIR, 'config')


# Make sure that the GPU is being used
# __________________________________________________________________________________________________
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda"

In [5]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
abalone = fetch_ucirepo(id=1) 
  
# data (as pandas dataframes) 
X = abalone.data.features 
y = abalone.data.targets

X = X.drop(columns=['Sex'], inplace=False)

df_data = pd.DataFrame(X, columns=abalone.data.feature_names)
df_data['target'] = y

scaler = MinMaxScaler()
df_data = pd.DataFrame(scaler.fit_transform(df_data), columns=df_data.columns)

X = df_data.drop(columns=['target'], inplace=False)
y = df_data['target']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Pre-denoising XAI Benchmarck

In [7]:
model_params = {
    'ridge': {"alpha": 1.0},
    'pls': {"n_components": 1},
    'tree': {"max_depth": 5},
    'svm': {"dual": 'auto'},
    'knn': {"n_neighbors": 5, "weights": 'uniform', "algorithm": 'auto', "leaf_size": 30, "p": 2, "n_jobs": None},
}
xai_benchmark = XAI_benchmark(is_ts = False, model_params = model_params)

In [8]:
xai_benchmark.fit(X_train, y_train.values.ravel())

Fitting Ridge model...
Fitting Partial Least Squares model...
Fitting Decision Tree model...
Fitting Support Vector Machine model...
Fitting K-Nearest Neighbours model...
All models fitted!


/mnt/homeGPU/JJavierAR/repsol_env/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [9]:
xai_benchmark.predict(X_test, y_test.values.ravel(), get_metrics=True)

({'ridge': array([ 0.38003037,  0.3275612 ,  0.43924189,  0.39214949,  0.34038061,
          0.33834242,  0.31818762,  0.27100008,  0.22968997,  0.34825369,
          0.38118863,  0.23758119,  0.37401718,  0.24148964,  0.27414924,
          0.40956867,  0.40843408,  0.33975017,  0.47395012,  0.26962057,
          0.64823593,  0.26758322,  0.36580726,  0.37264421,  0.30940389,
          0.25590614,  0.32409632,  0.26229435,  0.32014026,  0.30898338,
          0.33824979,  0.37167707,  0.22961759,  0.31457676,  0.29328786,
          0.29407342,  0.20137304,  0.12366368,  0.25910555,  0.21811942,
          0.39917119,  0.5677922 ,  0.40025956,  0.30591948,  0.35541355,
          0.38227802,  0.30652937,  0.29525298,  0.38921719,  0.15909516,
          0.21778531,  0.60732457,  0.32468314,  0.14615722,  0.21936232,
          0.34905272,  0.35931662,  0.45979775,  0.3367447 ,  0.22910169,
          0.38059801,  0.44458516,  0.3436406 ,  0.24923187,  0.25361468,
          0.35727369,  0.1623

In [21]:
# Transform the data into a tensor
train_dataset = TensorDataset(
    x=X_train.values,
    y=X_train.values
)
# Transform the data into a tensor
val_dataset = TensorDataset(
    x=X_test.values,
    y=X_test.values
)

# Create the dataloaders
batch_size = 64
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

In [22]:
denoiser = DenoisingAutoencoder(
    input_dim=X_train.shape[1],
    latent_dim=32,
).to(device)

In [23]:
# Set model parameters and create the model Trainer object
lr = 0.001
criterion = nn.MSELoss()
optimizer = optim.Adam(denoiser.parameters(), lr=lr)

# Define the trainer
trainer_basic = Trainer(
    model=denoiser,
    train_generator=train_dataloader,
    val_generator=train_dataloader,
    device=device,
    criterion=criterion,
    optimizer=optimizer,
    epoch_scheduler=None,
    batch_scheduler=None,
    patience=15,
    epochs=500,
    checkpoints_path=os.path.join(
        CHECKPOINT_PATH, f'abalone_denoising_autoencoder'
    ),
)

In [24]:
# denoiser = torch.load(os.path.join(CHECKPOINT_PATH, 'denoising_autoencoder.pth'))

In [25]:
# Train the model
model, train_losses, val_losses, best_train_loss, best_val_loss = trainer_basic.fit(verbose=True)


Training is started!


Epochs loop:  42%|▍| 208/500 [04:24<06:10,  1.27s/epoch, Train loss: 9.322045806579183e-05 - Val loss: 9.309259758

Training stopped as validation loss did not improve for 15 epochs.


In [34]:
y_pred_test = model(torch.tensor(X_test.values).float().to(device)).cpu().detach().numpy()

In [35]:
y_pred_test.shape

(836, 7)

In [36]:
# Show the metrics
gt_values = X_test.values
predicted_values = y_pred_test

mae = mean_absolute_error(gt_values, predicted_values)
mse = mean_squared_error(gt_values, predicted_values)
rmse = np.sqrt(mse)
# mape = np.mean(np.abs(gt_values - predicted_values) / gt_values) * 100
r_squared = r2_score(gt_values, predicted_values)
nn_benchmark = {
    'nn': {
        "MSE: ": mse,
        "RMSE: ": rmse,
        "MAE: ": mae,
        # "MAPE: ": mape,
        "R2: ": r_squared
    }
}
nn_benchmark

{'nn': {'MSE: ': 0.00012745087273352153,
  'RMSE: ': 0.011289414189120777,
  'MAE: ': 0.0066230626539112115,
  'R2: ': 0.9766274869126886}}